In [ ]:
# Run under the virtual environment at .venv/ in the batch-infer directory
# The batch-infer script should be included in the PATH after activating the environment, e.g. `source .venv/bin/activate`
!which batch-infer

/cluster/project/beltrao/jjaenes/26.01_batch-infer_foldx/.venv/bin/batch-infer


In [ ]:
# Use human & yeast pre-computed MSAs by specifying data-sources:
!cat config.yaml


alphafold3:
  data_sources: >-
    --data_dir=/cluster/project/beltrao/jjaenes/25.06.03_batch-infer/results/alphafold3_yeast/alphafold3_msas
    --data_dir=/cluster/work/beltrao/jjaenes/25.04.02_batch-infer-projects/af3_human/alphafold3_msas


In [2]:
# Create two input JSON files with a short human protein with a pre-computed MSAs (atox1)
# Add the same protein (atox1) with 2 or 5 glycines added at the end as "missing" chains that do not have a pre-computed MSA
!af3io input-create alphafold3_jsons/atox1_atox1gly5.json \
    --sequence MPKHEFSVDMTCGGCAEAVSRVLNKLGGVKYDIDLPNKKVCIESEHSMDTLLATLKKTGKTVSYLGLE \
    --sequence MPKHEFSVDMTCGGCAEAVSRVLNKLGGVKYDIDLPNKKVCIESEHSMDTLLATLKKTGKTVSYLGLEGGGGG
!af3io input-create alphafold3_jsons/atox1_atox1gly2_atox1gly5.json \
    --sequence MPKHEFSVDMTCGGCAEAVSRVLNKLGGVKYDIDLPNKKVCIESEHSMDTLLATLKKTGKTVSYLGLE \
    --sequence MPKHEFSVDMTCGGCAEAVSRVLNKLGGVKYDIDLPNKKVCIESEHSMDTLLATLKKTGKTVSYLGLEGG \
    --sequence MPKHEFSVDMTCGGCAEAVSRVLNKLGGVKYDIDLPNKKVCIESEHSMDTLLATLKKTGKTVSYLGLEGGGGG
# input JSONs stored under alphafold3_jons/
!ls -l alphafold3_jsons/

Setting name to: atox1_atox1gly5
Write:	/cluster/project/beltrao/jjaenes/26.01_batch-infer_foldx/results/alphafold3_datafill/alphafold3_jsons/atox1_atox1gly5.json
Setting name to: atox1_atox1gly2_atox1gly5
Write:	/cluster/project/beltrao/jjaenes/26.01_batch-infer_foldx/results/alphafold3_datafill/alphafold3_jsons/atox1_atox1gly2_atox1gly5.json
total 8
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 637 Jan 28 12:24 atox1_atox1gly2_atox1gly5.json
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 475 Jan 28 12:24 atox1_atox1gly5.json


In [3]:
# Find missing sequences (./ means use the current directory)
!batch-infer alphafold3_datafill_missing ./ | sbatch

Submitted batch job 55462125


In [ ]:
# For every missing sequence (atox1 with 2 or 5 glycines), there's now an input JSON under alphafold3_missing/
# The file names consists of the original input JSON, and the id of the missing sequence
# e.g. atox1_atox1gly2_atox1gly5_b refers to sequence B from alphafold3_jsons/atox1_atox1gly2_atox1gly5.json
# Duplicate missing chains are handled correctly, i.e. atox1gly5 appears in two input JSONs but only gets one "missing" JSON
!ls -l alphafold3_missing/

total 8
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 334 Jan 28 12:25 atox1_atox1gly2_atox1gly5_b.json
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 327 Jan 28 12:25 atox1_atox1gly5_b.json


In [5]:
# Run data pipeline only for the missing sequences
!batch-infer alphafold3_datafill_msas ./ | sbatch

Submitted batch job 55462443


In [8]:
# One job per every missing chain/file under alphafold3_missing/
!squeue --format="%.18i %.12P %.128j %.8T %.16M %.16l %40R" | column -t --table-right 1,5,6 | grep alphafold3_msas

55462453  normal.4h    alphafold3_msas:id=atox1_atox1gly5_b                                                   RUNNING        2:54     4:00:00  eu-a2p-337             
55462455  normal.4h    alphafold3_msas:id=atox1_atox1gly2_atox1gly5_b                                         RUNNING        2:54     4:00:00  eu-a2p-151             


In [9]:
# Data pipeline output for missing sequences stored under alphafold3_msas/
!ls -l alphafold3_msas/

total 772
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 423232 Jan 28 12:47 atox1_atox1gly2_atox1gly5_b_data.json.gz
-rw-r--r-- 1 jjaenes biol-imsb-beltrao 353389 Jan 28 12:46 atox1_atox1gly5_b_data.json.gz


In [11]:
# Run predictions, creating temporary data pipeline output on local scratch as-needed
!batch-infer alphafold3_datafill_predictions ./ | sbatch

Submitted batch job 55464713


In [ ]:
# Predictions stored as zip-compressed archives under alphafold3_predictions/
!ls -l alphafold3_predictions/

In [ ]:
# Cleanup
#!rm -rf .snakemake/
#!rm -rf .snakemake-eu/
#!rm -rf alphafold3_jsons
#!rm -rf alphafold3_missing
#!rm -rf alphafold3_msas
#!rm -rf alphafold3_predictions